# Eval Overview - Uncertainty Calibration

The uncertainty calibartion evaluation are a set of benchmarks that evaluate a particularly unique output from BioJEPA-AC, the *logvar*. Recall that when we run a forward inference pass of the model, we ouput the average latent representation, mu, and the variance, logvar.  The variance can tell us about the model's certainty in the prediction across the embedding and gene dimension.  This uncertainty output is a key unique feature of BioJEPA we have not yet seen in the space.  

Our eval suite on uncertainty first runs a series of sample level evaluations and then groups our prediction by perturbation for the remaining correlation assessment. We perform these on the full test set and provide the results subset by gene.  As you walk through the notebook you'll see that we evaluate on multiple dimensions to ensure we have a thorough understanding on where our model is working well and where it's struggling.

In [1]:
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

In [2]:
SEED = 1337
np.random.seed(SEED)

## Data Prep
We'll start by preparing our data. For this evaluation we need to have our change in expression, or *expression delta*, and our predicted logvar by sample. To get our expression delta, we need the true cell expressions, the control expression for each cell, and the perturbation information. We'll mock up 6 samples across 4 perturbations. A "perturbation" is a unique combination of a sequence, target, modality, and mode applied to a cell type. A perturbation can span datasets. For the 6 samples we'll show the real control cell expression, predicted control cell expression (running the student encoder and then linear decoder), real case cell expression, predicted case cell expression, the logvar, and the perturbation. We'll go ahead and do all 6 samples in batches.

For predicted expression, we use the data created by the [linear expression decoder](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_eval_decoders.ipynb). As a reminder, the decoder takes the mean ($\mu$) BioJEPA-AC output based on the perturbations and cell expression pattern, and then uses a linear layer to project down to a $[\text{n\_genes},\text{1}]$ matrix with a single value per gene representing the expression.

We'll stage data to show a few different predictions: a strong prediction, weak prediction, inverse prediction, and over prediction.

In [11]:
unique_perts = 4
num_genes = 8
num_cells = 6
embedding_dim=4

**Perturbations**

We'll first start with our perturbations. Even though we have 6 cells, our sample will only have 4 unique perturbations. To define a unique perturbation, it's not just about what we target, but the context of it. Because of this, we represent a unique perturbation as $\text{(seq id, targ id, modality id, mode id, cell type)}$. IDs are used since our model keeps the perturbation information in separate caches from our sample expression counts to avoid heavy duplication of information.

We'll also create a mapping of the sample to the perturbation, where the first two samples map to the first perturbation, the next two samples to the second perturbation, and then the final two samples each have a unique perturbation.

In [5]:
pert_keys = [                                                                      
      (0, 0, 0, 0, 0),  # pert 0: DNA CRISPRi, cell type 0 
      (1, 1, 0, 0, 0),  # pert 1: DNA CRISPRi, cell type 0 
      (2, 2, 0, 1, 0),  # pert 2: DNA CRISPRa, cell type 0 
      (3, 3, 2, 4, 0),  # pert 3: Chemical inhibitor, cell type 0
]


sample_to_pert = [0, 0, 1, 1, 2, 3]

**Control Cells**

Next we'll show the control cell values. Recall that for our inference we pair together a perturbed cell with a random control cell from the same batch. This allows us to take an approximate change in prediction. Beyond just the control cell expression, to calculate our predicted change in expression, we run the expression predictor on the control cell's latent representation `z_context`. We'll discuss the calculation more but to show this we'll also have the predicted control expression.

For the predicted, we'll show a minor shift to highlight that often the prediction is not perfect.

In [7]:
real_control = np.array([
    [2.1, 3.4, 1.2, 4.1, 2.6, 3.1, 1.4, 4.6], 
    [1.9, 3.6, 0.8, 3.9, 2.4, 2.9, 1.6, 4.4], 
    [2.0, 3.3, 1.1, 4.2, 2.3, 3.2, 1.3, 4.3], 
    [2.2, 3.7, 0.9, 3.8, 2.7, 2.8, 1.7, 4.7], 
    [2.0, 3.5, 1.0, 4.0, 2.5, 3.0, 1.5, 4.5], 
    [2.0, 3.5, 1.0, 4.0, 2.5, 3.0, 1.5, 4.5], 
])
pred_control = np.array([
    [2.0, 3.3, 1.3, 4.0, 2.5, 3.2, 1.3, 4.5], 
    [1.8, 3.5, 0.9, 3.8, 2.3, 3.0, 1.5, 4.3], 
    [1.9, 3.2, 1.2, 4.1, 2.2, 3.3, 1.2, 4.2], 
    [2.1, 3.6, 1.0, 3.7, 2.6, 2.9, 1.6, 4.6], 
    [1.9, 3.4, 1.1, 3.9, 2.4, 3.1, 1.4, 4.4], 
    [1.9, 3.4, 1.1, 3.9, 2.4, 3.1, 1.4, 4.4], 
])
real_control.shape, pred_control.shape

((6, 8), (6, 8))

**Case Cell**

Next we'll show the case cell values. In our raw data we have the real expression of the perturbed cell. We pair this together with the linear expression decoder output based on the mean prediction, $\mu$, from the ACPredictor output. The mean prediction is based on the control cell latent representation `z_context` and the perturbations. You'll quickly be able to see that there is a difference between the real values and the predicted values. We've staged the data so that the first two samples predict closely, the next two samples weakly, the fifth sample predicts in the wrong direction, and the final sample over predicts. You'll see how these calculations flow through.

In [8]:
real_case = np.array([
    [2.8, 2.3, 1.6, 6.0, 2.0, 4.7, 1.2, 2.9], 
    [2.8, 2.2, 1.0, 6.0, 2.0, 4.3, 1.6, 2.5], 
    [2.1, 3.2, 1.2, 4.1, 2.4, 3.3, 1.3, 4.4], 
    [2.3, 3.6, 0.9, 3.7, 2.7, 2.8, 1.7, 4.7], 
    [2.5, 2.7, 1.6, 3.0, 2.8, 3.7, 1.1, 5.7], 
    [2.3, 3.1, 1.2, 4.5, 2.2, 3.6, 1.4, 4.9], 
])

pred_case = np.array([
    [2.6, 2.3, 1.8, 5.8, 2.0, 4.6, 1.0, 3.0],  # strong
    [2.6, 2.3, 1.2, 5.8, 2.0, 4.5, 1.4, 2.6],  # strong
    [2.05, 3.05, 1.25, 4.05, 2.3, 3.35, 1.15, 4.3],  # weak
    [2.15, 3.6, 0.95, 3.6, 2.65, 2.95, 1.6, 4.65],  # weak
    [1.6, 3.9, 1.6, 4.5, 2.2, 3.9, 1.7, 3.9],  # inverse
    [2.8, 2.2, 1.7, 5.4, 1.5, 4.9, 1.1, 5.6],  # over
])

real_case.shape, pred_case.shape

((6, 8), (6, 8))

**Predicted Sample LogVar**

A key component of our uncertainty calibration is evaluating the log-variance that our model predicts.  The logvar is one of outputs from the ACPredictor based on the input control latent and the perturbation latent. The logvar dimensions match the cell state latents as $[\text{batch},\text{n\_genes},\text{embedding\_dim}]$. Since we'll be analyzing this against actual expression delta, we need to collapse the embedding dimension down to a single value per gene. We'll do this by taking the mean. This will then tell us, on average, how much variance the model expects the gene to have, a value that can tell us uncertainty by gene.

You'll notice that we seed the data with offsets $[-0.2, -0.1, 0.1, 0.2]$ around each target mean to make our calculations traceable. The real data is nowhere as clean. We've also setup variances to show high confidence (negatives) and low confidence (positives).  Reminder that this is log variance so the lower the number the better. 

In [20]:
pred_logvar=np.array([
      [[-2.2,-2.1,-1.9,-1.8],[-2.0,-1.9,-1.7,-1.6],[-2.1,-2.0,-1.8,-1.7],[-1.9,-1.8,-1.6,-1.5],[-2.3,-2.2,-2.0,-1.9],[-1.8,-1.7,-1.5,-1.4],[-2.2,-2.1,-1.9,-1.8],[-1.7,-1.6,-1.4,-1.3]],
      [[0.0,0.1,0.3,0.4],[0.1,0.2,0.4,0.5],[-0.1,0.0,0.2,0.3],[0.2,0.3,0.5,0.6],[-0.2,-0.1,0.1,0.2],[0.1,0.2,0.4,0.5],[-0.3,-0.2,0.0,0.1],[0.3,0.4,0.6,0.7]],
      [[0.1,0.2,0.4,0.5],[0.2,0.3,0.5,0.6],[-0.1,0.0,0.2,0.3],[0.4,0.5,0.7,0.8],[0.0,0.1,0.3,0.4],[0.3,0.4,0.6,0.7],[-0.2,-0.1,0.1,0.2],[0.4,0.5,0.7,0.8]],
      [[-2.1,-2.0,-1.8,-1.7],[-1.9,-1.8,-1.6,-1.5],[-2.0,-1.9,-1.7,-1.6],[-2.2,-2.1,-1.9,-1.8],[-2.0,-1.9,-1.7,-1.6],[-2.1,-2.0,-1.8,-1.7],[-2.0,-1.9,-1.7,-1.6],[-1.9,-1.8,-1.6,-1.5]],
      [[-2.7,-2.6,-2.4,-2.3],[-2.5,-2.4,-2.2,-2.1],[-2.2,-2.1,-1.9,-1.8],[-2.8,-2.7,-2.5,-2.4],[-2.4,-2.3,-2.1,-2.0],[-2.3,-2.2,-2.0,-1.9],[-2.6,-2.5,-2.3,-2.2],[-2.9,-2.8,-2.6,-2.5]],
      [[0.0,0.1,0.3,0.4],[0.3,0.4,0.6,0.7],[-0.3,-0.2,0.0,0.1],[0.6,0.7,0.9,1.0],[0.1,0.2,0.4,0.5],[0.4,0.5,0.7,0.8],[-0.4,-0.3,-0.1,0.0],[0.2,0.3,0.5,0.6]],
])
pred_logvar.shape, pred_logvar

((6, 8, 4),
 array([[[-2.2, -2.1, -1.9, -1.8],
         [-2. , -1.9, -1.7, -1.6],
         [-2.1, -2. , -1.8, -1.7],
         [-1.9, -1.8, -1.6, -1.5],
         [-2.3, -2.2, -2. , -1.9],
         [-1.8, -1.7, -1.5, -1.4],
         [-2.2, -2.1, -1.9, -1.8],
         [-1.7, -1.6, -1.4, -1.3]],
 
        [[ 0. ,  0.1,  0.3,  0.4],
         [ 0.1,  0.2,  0.4,  0.5],
         [-0.1,  0. ,  0.2,  0.3],
         [ 0.2,  0.3,  0.5,  0.6],
         [-0.2, -0.1,  0.1,  0.2],
         [ 0.1,  0.2,  0.4,  0.5],
         [-0.3, -0.2,  0. ,  0.1],
         [ 0.3,  0.4,  0.6,  0.7]],
 
        [[ 0.1,  0.2,  0.4,  0.5],
         [ 0.2,  0.3,  0.5,  0.6],
         [-0.1,  0. ,  0.2,  0.3],
         [ 0.4,  0.5,  0.7,  0.8],
         [ 0. ,  0.1,  0.3,  0.4],
         [ 0.3,  0.4,  0.6,  0.7],
         [-0.2, -0.1,  0.1,  0.2],
         [ 0.4,  0.5,  0.7,  0.8]],
 
        [[-2.1, -2. , -1.8, -1.7],
         [-1.9, -1.8, -1.6, -1.5],
         [-2. , -1.9, -1.7, -1.6],
         [-2.2, -2.1, -1.9, -1.8],

In [21]:
sample_logvar = pred_logvar.mean(axis=-1)
sample_logvar.shape, sample_logvar

((6, 8),
 array([[-2.0000000e+00, -1.8000000e+00, -1.9000000e+00, -1.7000000e+00,
         -2.1000000e+00, -1.6000000e+00, -2.0000000e+00, -1.5000000e+00],
        [ 2.0000000e-01,  3.0000000e-01,  1.0000000e-01,  4.0000000e-01,
         -6.9388939e-18,  3.0000000e-01, -1.0000000e-01,  5.0000000e-01],
        [ 3.0000000e-01,  4.0000000e-01,  1.0000000e-01,  6.0000000e-01,
          2.0000000e-01,  5.0000000e-01, -6.9388939e-18,  6.0000000e-01],
        [-1.9000000e+00, -1.7000000e+00, -1.8000000e+00, -2.0000000e+00,
         -1.8000000e+00, -1.9000000e+00, -1.8000000e+00, -1.7000000e+00],
        [-2.5000000e+00, -2.3000000e+00, -2.0000000e+00, -2.6000000e+00,
         -2.2000000e+00, -2.1000000e+00, -2.4000000e+00, -2.7000000e+00],
        [ 2.0000000e-01,  5.0000000e-01, -1.0000000e-01,  8.0000000e-01,
          3.0000000e-01,  6.0000000e-01, -2.0000000e-01,  4.0000000e-01]]))

**Calculate Sample Delta**

A major component of our expression benchmark is not looking at absolute predictions, but the change in expression. Some claim that this simplifies the task. Biologically, we see this as addressing the important questions: can you predict what will change, in what direction, and by how much. We focus on calculating two sample level differences:
1. `pred_delta` - the predicted change in expression as calculated by $\hat{\delta}_g = \hat{x}^{\text{case}}_g - \hat{x}^{\text{ctrl}}_g$. This value compares the predicted perturbed expression (`pred_case`) from the predicted control expression (`pred_control`). We use the predicted control expression to isolate BioJEPA-AC's learned perturbation effect from any baseline reconstruction error.
2. `real_delta` - the real change in expression as calculated by $\delta_g = x^{\text{case}}_g - x^{\text{ctrl}}_g$. This is our source of truth.

With this calculation you'll see how we end up seeing both increases and decreases in expression by gene. We'll end up comparing these by different slices in our calculations.

In [22]:
pred_delta = pred_case - pred_control

pred_delta.shape, pred_delta

((6, 8),
 array([[ 0.6 , -1.  ,  0.5 ,  1.8 , -0.5 ,  1.4 , -0.3 , -1.5 ],
        [ 0.8 , -1.2 ,  0.3 ,  2.  , -0.3 ,  1.5 , -0.1 , -1.7 ],
        [ 0.15, -0.15,  0.05, -0.05,  0.1 ,  0.05, -0.05,  0.1 ],
        [ 0.05,  0.  , -0.05, -0.1 ,  0.05,  0.05,  0.  ,  0.05],
        [-0.3 ,  0.5 ,  0.5 ,  0.6 , -0.2 ,  0.8 ,  0.3 , -0.5 ],
        [ 0.9 , -1.2 ,  0.6 ,  1.5 , -0.9 ,  1.8 , -0.3 ,  1.2 ]]))

In [23]:
real_delta = real_case - real_control

real_delta.shape, real_delta

((6, 8),
 array([[ 0.7, -1.1,  0.4,  1.9, -0.6,  1.6, -0.2, -1.7],
        [ 0.9, -1.4,  0.2,  2.1, -0.4,  1.4,  0. , -1.9],
        [ 0.1, -0.1,  0.1, -0.1,  0.1,  0.1,  0. ,  0.1],
        [ 0.1, -0.1,  0. , -0.1,  0. ,  0. ,  0. ,  0. ],
        [ 0.5, -0.8,  0.6, -1. ,  0.3,  0.7, -0.4,  1.2],
        [ 0.3, -0.4,  0.2,  0.5, -0.3,  0.6, -0.1,  0.4]]))

## Sample Level

Our first set of uncertainty evaluations all happens at the sample level. Since we're evaluating the uncertainty, even though samples can be noisy, we're trying to understand just how noisy the predictions can be. We'll compare uncertainty to our prediction error, we'll then analyze our uncertainty by binning it, and finally we'll analyze normalized uncertainty.

**Uncertainty**

We first need to convert our logvar into a single value per sample that we'll call *uncertainty*. We'll run a simple mean on the logvar using the calculation 
$$
\text{uncertainty}_\text{sample} = \frac{1}{G}\sum_{g=1}^{G} \text{logvar}_{g}  
$$

**Mean Squared Error**

Now we need to know what to compare it against.  The logvar should show the variance within which our model prediction is correct. To analyze this, we'll look at the mean squared error (MSE) of our predicted expression delta against the real expression delta. We calculate this as:
$$
\text{MSE}_\text{sample} = \frac{1}{G}\sum_{g=1}^{G} (\hat{\delta}_{g} - \delta_{g})^2   
$$


Our comparisons will then look to see how well our MSE aligns with our uncertainty.  We'll start by calulcating the sample uncertainty and MSE. You can see in the uncertainty which samples have high confidence (low uncertainty) and which don't. Similarly you can also see that while we have some samples with high uncertainty, their MSE is low showing they're uncertain but correct.

In [25]:
sample_unc = sample_logvar.mean(axis=1)
sample_unc.shape, sample_unc

((6,), array([-1.825 ,  0.2125,  0.3375, -1.825 , -2.35  ,  0.3125]))

In [26]:
sample_mse = np.mean((pred_delta - real_delta)**2, axis=1)
sample_mse.shape, sample_mse

((6,),
 array([0.0175   , 0.0175   , 0.001875 , 0.0028125, 1.0675   , 0.58     ]))